In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/devonchao/tripdata/202605-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202604-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202509-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202511-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202512-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202506-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202603-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202507-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202501-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202508-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202602-divvy-tripdata.csv
/kaggle/input/datasets/devonchao/tripdata/202510-divvy-tripdata.csv


In [3]:
import pandas as pd
import os

path = '/kaggle/input/datasets/devonchao/tripdata/'
files = [f for f in os.listdir(path) if f.endswith('.csv')]

# combine the 12 files
df = pd.concat([pd.read_csv(path + f) for f in files], ignore_index=True)

print(df.shape)
print(df.dtypes)
print(df.head())

(5848703, 13)
ride_id                object
rideable_type          object
started_at             object
ended_at               object
start_station_name     object
start_station_id       object
end_station_name       object
end_station_id         object
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual          object
dtype: object
            ride_id  rideable_type               started_at  \
0  E8847A2B0FD51251  electric_bike  2026-05-20 11:36:57.782   
1  746948BC4E29B6D0  electric_bike  2026-05-20 11:37:45.232   
2  33E3CE21C190985E  electric_bike  2026-05-20 18:37:05.606   
3  63D3912D5A15ED09  electric_bike  2026-05-20 08:00:52.487   
4  76EBC4E28ABC6A5E  electric_bike  2026-05-19 19:39:44.270   

                  ended_at start_station_name start_station_id  \
0  2026-05-20 11:37:17.020                NaN              NaN   
1  2026-05-20 11:51:43.399                NaN              NaN   
2  20

In [4]:
print(df.columns.tolist())
print(df.isnull().sum())

['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']
ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1249667
start_station_id      1249667
end_station_name      1314334
end_station_id        1314334
start_lat                   0
start_lng                   0
end_lat                  5896
end_lng                  5896
member_casual               0
dtype: int64


In [5]:
# convert to datetime first
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

# add ride_length in minutes
df['ride_length'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# add day of week (1=Monday, 7=Sunday in pandas)
df['day_of_week'] = df['started_at'].dt.day_name()

print(df['ride_length'].describe())

count    5.848703e+06
mean     1.598668e+01
std      5.510297e+01
min     -5.479480e+01
25%      5.372292e+00
50%      9.402117e+00
75%      1.654735e+01
max      1.574900e+03
Name: ride_length, dtype: float64


In [6]:
# drop negative ride lengths and extreme outliers
df = df[df['ride_length'] > 0]
df = df[df['ride_length'] <= 1440]

# drop rows with missing end coordinates
df = df.dropna(subset=['end_lat', 'end_lng'])

print(df.shape)
print(df['ride_length'].describe())

(5842578, 15)
count    5.842578e+06
mean     1.448544e+01
std      2.860753e+01
min      7.666667e-04
25%      5.368583e+00
50%      9.392008e+00
75%      1.651293e+01
max      1.439976e+03
Name: ride_length, dtype: float64


In [7]:
# average ride length by member type
print(df.groupby('member_casual')['ride_length'].mean())

# number of rides by member type
print(df.groupby('member_casual')['ride_length'].count())

# average ride length by member type and day of week
print(df.groupby(['member_casual', 'day_of_week'])['ride_length'].mean().unstack())

# number of rides by member type and day of week
print(df.groupby(['member_casual', 'day_of_week'])['ride_length'].count().unstack())

# bike type preference by member type
print(df.groupby(['member_casual', 'rideable_type'])['ride_id'].count().unstack())

member_casual
casual    18.832504
member    12.054806
Name: ride_length, dtype: float64
member_casual
casual    2095281
member    3747297
Name: ride_length, dtype: int64
day_of_week       Friday     Monday   Saturday     Sunday   Thursday  \
member_casual                                                          
casual         18.372203  18.995611  21.111373  21.819220  16.451682   
member         11.962222  11.802872  13.161190  13.223901  11.605278   

day_of_week      Tuesday  Wednesday  
member_casual                        
casual         16.447892  15.633899  
member         11.628537  11.554692  
day_of_week    Friday  Monday  Saturday  Sunday  Thursday  Tuesday  Wednesday
member_casual                                                                
casual         322621  244705    431688  359336    266530   234377     236024
member         548546  527106    469394  414246    601351   592384     594270
rideable_type  classic_bike  electric_bike
member_casual                     

In [8]:
# change bike preference+ to percentage 
bike_pct = df.groupby(['member_casual', 'rideable_type'])['ride_id'].count().reset_index().rename(columns={'ride_id': 'ride_count'})
bike_pct['total'] = bike_pct.groupby('member_casual')['ride_count'].transform('sum')
bike_pct['percentage'] = (bike_pct['ride_count'] / bike_pct['total'] * 100).round(1)
print(bike_pct)

  member_casual  rideable_type  ride_count    total  percentage
0        casual   classic_bike      657035  2095281        31.4
1        casual  electric_bike     1438246  2095281        68.6
2        member   classic_bike     1283048  3747297        34.2
3        member  electric_bike     2464249  3747297        65.8


In [9]:
# export for visualization
df.groupby(['member_casual', 'day_of_week'])['ride_length'].mean().reset_index().to_csv('avg_ride_length_by_day.csv', index=False)
df.groupby(['member_casual', 'day_of_week'])['ride_id'].count().reset_index().rename(columns={'ride_id': 'ride_count'}).to_csv('ride_count_by_day.csv', index=False)
bike_pct.to_csv('bike_type_preference.csv', index=False)

In [10]:
df_check = pd.read_csv('ride_count_by_day.csv')
print(df_check.dtypes)
print(df_check.head())

member_casual    object
day_of_week      object
ride_count        int64
dtype: object
  member_casual day_of_week  ride_count
0        casual      Friday      322621
1        casual      Monday      244705
2        casual    Saturday      431688
3        casual      Sunday      359336
4        casual    Thursday      266530
